## If working in Google Colab, run this first
Run the cell below at the start of every session. It downloads the course files to your Google Drive and keeps them up to date.
When you see "Setup complete", you are ready to run the rest of the notebook.

**Do not edit or save your work inside the `AIforMovementScience` folder.**
Use **File > Save a copy in Drive** and work on your own copy.

In [ ]:
# [course-setup] Run this cell first, every session (or use Runtime > Run all)
import os, sys
try:
    from google.colab import drive
    os.chdir('/content')                  # start from a valid directory
    drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/AIforMovementScience'
    if os.path.isdir(REPO + '/.git'):
        # already downloaded before: just update it
        !git -C "$REPO" fetch origin main && git -C "$REPO" reset --hard origin/main
    else:
        # first time, or a broken/partial copy: download a fresh copy
        !rm -rf "$REPO"
        !git clone https://github.com/dguari1/AIforMovementScience.git "$REPO"
except ImportError:
    # not in Colab: walk up from here until the repository root is found
    REPO = os.getcwd()
    while REPO != '/' and not os.path.isdir(os.path.join(REPO, 'lectures', 'data')):
        REPO = os.path.dirname(REPO)

WORKDIR = REPO + '/lectures'
os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)
print('Setup complete. You can now run the rest of the notebook.')

# CNN applications

This notebook uses the library **ultralytics** (YOLO models). It is not installed in Colab by default. Run the next cell once per session (it takes about 1 minute).

In [ ]:
!pip install -q ultralytics

## It starts with a dataset 

As discussed, in most application, you will be using CNN models that other people created and trained with different datasets. 

There are hundreds (thousands) of such models, each model has its own accuracy, number of parameters, processing speed, etc. 

See [TIMMS Leaderboard](https://huggingface.co/spaces/timm/leaderboard) for a current list of models.

For instance, here is the list of the best models on 04/07/2026
<div style="text-align:center">
<img src="images/imagenet_results.png" alt="plot" width="700" />
</div>

### ImageNet

To compare different models fairly, researchers developed a set of large databases that include images along with some desired output. The models are then trained on these large databases and different models are ranked according to the results they obtain for hold-out test set. Modern models are expected to be good at different tasks, so there are many available databases. These datasets are open-access, you can download them and test your models.

The first database of its type was **ImageNet.** 

ImageNet (the version used to compare models) has about 1.2 million training images and 1,000 categories (animals, objects, food, vehicles, etc.). Each image was labeled by people with the main object in the image.

<div style="text-align:center">
<img src="images/ImageNet_sample.png" alt="plot" width="600" />
</div>

These models have 1,000 outputs: one probability for each category. Two scores are used:
- **top-1 accuracy**: the fraction of test images where the most probable category is the correct one;
- **top-5 accuracy**: the fraction of test images where the correct category is one of the 5 most probable categories.

Many ImageNet categories are very similar (for example many breeds of dogs), so the top-5 accuracy does not penalize the model too much for confusing similar categories.

Here is a list of all 1000 ImageNet categories: https://deeplearning.cms.waikato.ac.nz/user-guide/class-maps/IMAGENET/

### COCO

The COCO (Common Objects in COntext) dataset is a **different** dataset from ImageNet. Its images show everyday scenes with **many objects**, and each object is labeled with its category and its **position** (a box) in the image. So COCO is used to train and test **object detection** models.

<div style="text-align:center">
<img src="images/COCOSample.png" alt="plot" width="400" />
</div>

The COCO dataset contains 330,000 images, with more than 1.5 million instances (multiple objects per image). There are 80 different objects in the dataset. 

Here is a list of all 80 COCO categories: https://github.com/ultralytics/ultralytics/blob/main/ultralytics/cfg/datasets/coco.yaml

Some categories include:
- 0: person
- 1: bicycle
- 2: car
- 3: motorcycle
- 4: airplane
- 5: bus
- 6: train
- 7: truck
- 8: boat
- 9: traffic light
- 10: fire hydrant
- 11: stop sign

### COCO segmentation

COCO also includes a **segmentation** of each object (the exact pixels of the object)
<div style="text-align:center">
<img src="images/COCOSegment.avif" alt="plot" width="400" />
</div>

## COCO Pose

COCO also includes the position of 17 **body keypoints** for each person (we use this in module 08)
<div style="text-align:center">
<img src="images/COCOPose.webp" alt="plot" width="400" />
</div>

### Learning Objective

The objective of this notebook is to learn :
- What a Convolutional Neural Network (CNN) does in real-world applications
- How pretrained models are used (instead of training from scratch)
- How object detection works at a high level
- How to interpret model outputs (boxes, labels, confidence)

For this, we use a family of models designed to work in real time, called **YOLO** (You Only Look Once). YOLO models give a good balance between speed and accuracy. There are YOLO models for different tasks: classification (scored on ImageNet), object detection, segmentation and pose estimation (scored on COCO). Each family has sizes from `n` (nano, the fastest and least accurate) to `s`, `m`, `l` and `x` (the slowest and most accurate).

<div style="text-align:center">
<img src="images/YOLOResults.png" alt="plot" width="600" />
</div>

 In this notebook, we will use an object detection model trained on the COCO dataset. The model:
- detects objects in an image (only objects included in the COCO dataset)
- draws bounding boxes
- assigns class labels (person, car, dog, etc.)
- gives confidence scores for each object

If you want higher precision, you can always use other models, but YOLO models are a good starting point. 

In [ ]:
import os
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
from ultralytics import YOLO
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import pandas as pd

In [ ]:
# load the pretrained detection model (trained on COCO). 's' = small: fast, a little less accurate than m, l or x
model = YOLO('../models/yolo26s.pt')

In [ ]:
model.info()   # number of layers and parameters

In [ ]:
#let's test the model on an image
image_path = r'data/cat-dog.jpg'
img = Image.open(image_path)   # PIL image (RGB)
plt.imshow(img) #show the image
plt.show()

When using YOLO models from Ultralytics, we can pass the image directly as it is. The model will take care of adjusting the image size. 

In [ ]:
#process the image with the model 
results = model(img)

### Understanding the `results` object

`results` is a **list** returned by YOLO inference.  
Since we passed one image (`img`), `results` has length **1**.

- `results[0]` is an object that stores the results for the image.
- Main attributes in `results[0]`:
    - `boxes`: detected bounding boxes (coordinates, confidence, class id)
    - `names`: mapping from class id to class name (COCO labels)
    - `orig_img`, `orig_shape`: original image data and size
    - `speed`: preprocessing / inference / postprocessing time
    - `masks`, `keypoints`, `probs`, `obb`: task-specific outputs (often `None` for standard detection)

For this notebook, the most important field is **`results[0].boxes`**, which contains the detected objects (e.g., cat, dog) with their confidence scores and box locations.

In [ ]:
results[0].boxes

### Information in `boxes`

`boxes = results[0].boxes` contains all detections for the image.

- `boxes.cls`: predicted class ID for each detection (COCO classes)
- `boxes.conf`: confidence score for each detection, a number between 0 and 1
- `boxes.xyxy`: bounding box coordinates in pixel format `[x1, y1, x2, y2]`
- `boxes.xywh`: the **center** of the box `[x, y]`, and the box width `[w]` and height `[h]`, in pixels.
- `boxes.xyxyn`: same as `boxes.xyxy` but normalized by the image width `[w]` and height `[h]`.
- `boxes.xywhn`: same as `boxes.xywh` but normalized by the image width `[w]` and height `[h]`.


This means each detection can be described as:

1. Object class (e.g., `cat`, `dog`)
2. Confidence score (between 0 and 1)
3. Bounding box location in the image

We can now plot the image and the results. We write a small function, because we will use it many times.

In [ ]:
def show_detections(img, results, names):
    plt.figure(figsize=(8, 6))
    plt.imshow(img)
    boxes = results[0].boxes.xyxy.cpu().numpy()      # [x1, y1, x2, y2] for each detection
    classes = results[0].boxes.cls.cpu().numpy()
    conf = results[0].boxes.conf.cpu().numpy()
    for box, c, p in zip(boxes, classes, conf):
        x1, y1, x2, y2 = box
        plt.gca().add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color='red', linewidth=2))
        plt.text(x1, y1, f'{names[int(c)]} {p:.2f}', color='red', fontsize=11)
    plt.axis('off')
    plt.show()

show_detections(img, results, model.names)

In [ ]:
# another image
img = Image.open('data/objects.jpg')
results = model(img)
show_detections(img, results, model.names)

## Why does the model make mistakes?

Common reasons:
- unusual angles
- occlusion (object partially hidden)
- lighting conditions
- **object not in training dataset**

### Activity 1

Load the image `data/playing.jpeg` and use the YOLO model to identify the objects in the image

In [ ]:
# Write your code here

## Time Up and Go (TUG) test

The folder `data/TUG` contains frames of a person who does the Timed Up and Go (TUG) test: stand up from a chair, walk 3 m, turn, walk back and sit down. The frames were saved from the video `data/TUG.mp4`. Let us look at a few frames.

In [ ]:
list_of_images = sorted(f for f in os.listdir('data/TUG') if f.endswith('.jpg'))   # only images (macOS creates hidden files)
print('Number of frames:', len(list_of_images))

plt.figure(figsize=(16, 3))
for k, idx in enumerate([0, 50, 150, 300]):
    plt.subplot(1, 4, k + 1)
    plt.imshow(Image.open('data/TUG/' + list_of_images[idx]))
    plt.title(f'frame {idx}'); plt.axis('off')
plt.show()

### Activity 2

Use the YOLO model on the first frame. What objects can you detect?

In [ ]:
img = Image.open('data/TUG/' + list_of_images[0])
results = model(img)
show_detections(img, results, model.names)

We can now use a loop to find the position of the person and the chair in each frame. We tell the model to look only for these two classes (`classes=...`), which is faster and avoids other detections. In each frame, we keep the detection with the highest confidence for each class, and we store the **center** of its box.

In [ ]:
name_to_id = {name: idx for idx, name in model.names.items()}
target_ids = [name_to_id['person'], name_to_id['chair']]

positions = []
for frame_number, fname in enumerate(list_of_images):
    r = model(os.path.join('data/TUG', fname), classes=target_ids, verbose=False)[0]
    frame_data = {'frame': frame_number, 'person_x': np.nan, 'person_y': np.nan, 'chair_x': np.nan, 'chair_y': np.nan}
    if r.boxes is not None and len(r.boxes) > 0:
        cls = r.boxes.cls.cpu().numpy().astype(int)
        conf = r.boxes.conf.cpu().numpy()
        xyxy = r.boxes.xyxy.cpu().numpy()
        for label in ['person', 'chair']:
            rows = np.where(cls == name_to_id[label])[0]
            if rows.size > 0:
                best = rows[np.argmax(conf[rows])]              # highest-confidence detection of this class
                x1, y1, x2, y2 = xyxy[best]
                frame_data[f'{label}_x'] = (x1 + x2) / 2         # center of the box
                frame_data[f'{label}_y'] = (y1 + y2) / 2
    positions.append(frame_data)

df = pd.DataFrame(positions)
df.head()

Plot the position of the person against the frame number.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(df['frame'], df['person_x']); ax[0].set_title('Person: horizontal position')
ax[1].plot(df['frame'], df['person_y']); ax[1].set_title('Person: vertical position')
ax[1].invert_yaxis()   # image coordinates: y increases downwards, so we invert the axis (up = up)
for a in ax:
    a.set_xlabel('Frame number'); a.set_ylabel('Position (pixels)')
plt.show()

**Questions**
- Use the plots to find the frame where the person starts to stand up and the frame where the person is seated again. How many frames does the test take?
- To get the duration in **seconds**, you need the time between frames. In notebook 07.02a, you will read the video directly and get the time of each frame.
- Where do the detections fail? Why?